In [7]:
import sys
sys.path.append("/home/hkerino/Documents/next_oso")
from data_functions import dataset
from Alise import ALISE, utils_ALISE, sse, temporal_proj, transformer, master_query_decoding
from einops import rearrange, repeat

import importlib

PATH_TO_DATA_s2 = "/home/hkerino/Documents/nextoso-dataset/data/10m/s2"
PATH_MNT = "/home/hkerino/Documents/nextoso-dataset/data/10m/mnt"
PATH_TO_DATA_label = "/home/hkerino/Documents/nextoso-dataset/data/10m/labels"
PATH_TO_OSO_CLASSES = (
    "/home/hkerino/Documents/nextoso-dataset/data/auxiliary/OSO/oso_classes.csv"
)

In [2]:
batch_size = 1

dl_train, dl_val, dl_test = dataset.get_loaders(
    path_s2=PATH_TO_DATA_s2,
    path_mnt=PATH_MNT,
    path_label=PATH_TO_DATA_label,
    path_oso_class=PATH_TO_OSO_CLASSES,
    folds_train=[1],
    folds_val=[4],
    folds_test=[5],
    batch_size=batch_size,
    num_worker=4,
    norm=False,
    norm_alise=True,
    shuffle=True,
    label_type="max_class",
    use_spatial_coords=False,
    mask_needed=False,
    sequence_packing=False,
    pass_id=False,
)

In [3]:
inputs, label = next(iter(dl_train))
print(inputs[0].shape)
print(inputs[1].shape)

torch.Size([1, 10, 25, 128, 128])
torch.Size([1, 25])


## Spatial encoding

In [4]:
x_in = inputs[0].permute(0, 2, 1, 3, 4).contiguous()
B, T, C, H, W = x_in.shape

#U-NET
unet = sse.Unet(planes=64)
x = rearrange(x_in, "B T C H W -> (B T) C H W ")
out_unet = unet(x)
out_unet =  rearrange(out_unet, "(B T) C H W -> B T C H W", B=B, T=T, H=H, W=W)
# Positional Encoding
pe = sse.PositionalEncoder(d=64)
out_pe = pe(inputs[1])
out_pe = repeat(out_pe, 'B T C -> B T C H W', H=H, W=W)

# Addition
out = out_unet + out_pe
print(out.shape)

#As one bloc
spatial_encoder = sse.PatchEmbedding(sse= unet, tpe=pe)
out_se = spatial_encoder(x_in, inputs[1])
print(out_se.shape)

(out == out_se).all()

torch.Size([1, 49, 64, 128, 128])
torch.Size([1, 49, 64, 128, 128])


tensor(True)

## Temporal encoding

In [5]:
x_in = rearrange(out_se, "B T C H W -> (B H W ) T C")
pad_mask = repeat(inputs[1] == 0, "B T -> (B H W) T", H=H, W=W) #create mask where dates "0" are flagged True

temporal_encoder = transformer.Transformer(d_model=64)
out_temp = temporal_encoder(x_in, src_key_padding_mask=pad_mask)
out_temp.shape

torch.Size([16384, 49, 64])

## Temporal projection

In [21]:
proj_temp = temporal_proj.LearnedQMultiHeadAttention(n_head=4,d_in=64, d_k=32, n_q=5)
out_proj = proj_temp(out_temp, pad_mask=~pad_mask)
print(out_proj.shape)

torch.Size([16384, 5, 64])


## Decoding

In [26]:
mlp = utils_ALISE.MLP_ONNX(mlp=[5*64,3*64,64,25])
out = mlp(out_proj)
print(out.shape)

torch.Size([16384, 25])


## Master query decoding

In [14]:
inputs[1] == 0

tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False]])

In [9]:
decoder = master_query_decoding.MasterQueryDecoding(inplanes=64, planes=25, nhead=1)
in_decoder = rearrange(out_temp, "(B H W ) T C -> B T C H W", B=B, H=H, W=W)
out_decoder = decoder(in_decoder, key_padding_mask= (inputs[1]==0))

In [11]:
print(in_decoder.shape)
out_decoder.shape

torch.Size([1, 49, 64, 128, 128])


torch.Size([1, 128, 128, 25])

In [13]:
import torch
key_padding_mask = inputs[1]
print(key_padding_mask.shape)
n_head=3

key_padding_mask = torch.unsqueeze(key_padding_mask, dim=1)
key_padding_mask = torch.repeat_interleave(
    key_padding_mask, repeats=H * W, dim=1
)
print(key_padding_mask.shape)

key_padding_mask = rearrange(key_padding_mask, "b hw n -> (b hw ) n 1")
key_padding_mask = torch.repeat_interleave(
    key_padding_mask, repeats=n_head, dim=2
)
print(key_padding_mask.shape)

key_padding_mask = rearrange(key_padding_mask, "bs n c -> (bs c) 1 n ")

torch.Size([1, 32])
torch.Size([1, 16384, 32])
torch.Size([16384, 32, 3])


In [14]:
key_padding_mask.shape

torch.Size([49152, 1, 32])

## ALISE + MasterQueryDecoding

In [12]:
importlib.reload(ALISE)
importlib.reload(master_query_decoding)

#ALISE Encoding
unet = sse.Unet(planes=64)
pe = sse.PositionalEncoder(d=64)
spatial_encoder = sse.PatchEmbedding(sse=unet, tpe=pe)
temporal_encoder = transformer.Transformer(d_model=64)
encoder = ALISE.ALISE_encoding(patch_embedding=spatial_encoder, temporal_encoder=temporal_encoder)

embedding = encoder(inputs[0], inputs[1])

# Embedding decoding
decoder = master_query_decoding.MasterQueryDecoding(inplanes=64, planes=25, nhead=1)
out = decoder(embedding, inputs[1]==0)

In [13]:
print(embedding.shape)
print(out.shape)

torch.Size([1, 25, 64, 128, 128])
torch.Size([16384, 25])
